#WikipediaRetriever


In [2]:
# install packages (langchain-classic holds EnsembleRetriever, MultiQueryRetriever, ContextualCompressionRetriever)
!pip install -q langchain-chroma langchain-huggingface sentence-transformers langchain-community rank_bm25 langchain-classic


In [22]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

vector_store = Chroma(
    embedding_function = embeddings,
    collection_name = "players_retrieval",
)
vector_store.add_documents(docs, ids = ['babar', 'shaheen', 'rizwan', 'shadab', 'rauf'])

def show(retriever, query):
    print("Query:", query)
    for d in retriever.invoke(query):
        label = d.metadata.get("name") or d.metadata.get("title") or "(no title)"
        extra = d.metadata.get("role", "")
        print("  -", label, "|", extra, "|", d.page_content[:80].replace("\n", " "))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [23]:
!pip install -q wikipedia langchain-community
from langchain_community.retrievers import WikipediaRetriever

In [24]:

wiki_retriever= WikipediaRetriever(
    lang = "en",
    top_k_results = 2,
    doc_content_chars_max=2000,
)

show(wiki_retriever, 'Babar Azam')




Query: Babar Azam
  - Babar Azam |  | Mohammad Babar Azam (Urdu, Punjabi: محمد بابر اعظم; pronounced [muhamməd babəɾ a
  - List of international cricket centuries by Babar Azam |  | Babar Azam is a Pakistani  cricketer. As of 15 November 2025, he has played 61 T


## 1. Inspect what the retriever returns
Each Wikipedia page is one Document. Wikipedia metadata has `title`, `summary` and `source` (not `name`/`role`).

In [ ]:
wiki_docs = wiki_retriever.invoke("Babar Azam")

print("Documents returned:", len(wiki_docs))
for d in wiki_docs:
    print("\nTitle :", d.metadata.get("title"))
    print("Source:", d.metadata.get("source"))
    print("Metadata keys:", list(d.metadata.keys()))
    print("Length:", len(d.page_content), "characters")
    print(d.page_content[:300])

## 2. Try several queries

In [ ]:
for q in ["Shaheen Afridi", "Pakistan national cricket team", "Retrieval-augmented generation"]:
    show(wiki_retriever, q)
    print()

## 3. Query style matters: short topic vs long question
Wikipedia search matches titles and keywords, so a short topic usually works better.

In [ ]:
show(wiki_retriever, "Who is the best fast bowler of Pakistan?")   # long question
print()
show(wiki_retriever, "Pakistan fast bowlers")                      # short topic

## 4. Use it in a RAG chain (needs an LLM)
We pass the **topic** to the retriever and the **question** to the prompt.

In [ ]:
!pip install -q langchain-google-genai

In [ ]:
import os
from getpass import getpass
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = getpass("Google AI Studio API key: ")
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)   # change to a current model name if needed

In [ ]:
from operator import itemgetter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(f"{d.metadata.get('title')}:\n{d.page_content}" for d in docs)

prompt = PromptTemplate.from_template(
    "Answer using ONLY the context below. If the answer is not in the context, say you don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

wiki_chain = (
    {
        "context": itemgetter("topic") | wiki_retriever | format_docs,
        "question": itemgetter("question"),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print(wiki_chain.invoke({
    "topic": "Babar Azam",
    "question": "Which cricket formats is Babar Azam known for, and what is his role in the team?",
}))

## 5. Better for long pages: split the Wikipedia pages, store the chunks, then retrieve the best chunks
Whole pages are large. Chunking sends the LLM only the relevant parts.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

big_wiki = WikipediaRetriever(lang="en", top_k_results=3, doc_content_chars_max=10000)
pages = big_wiki.invoke("Pakistan national cricket team")
print("Pages fetched:", [p.metadata.get("title") for p in pages])

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)
print("Chunks:", len(chunks))

wiki_store = Chroma(embedding_function=embeddings, collection_name="wiki_chunks")
wiki_store.add_documents(chunks)

chunk_retriever = wiki_store.as_retriever(search_kwargs={"k": 3})
show(chunk_retriever, "Who has captained the Pakistan team?")

## Summary
- `WikipediaRetriever` searches Wikipedia live, so nothing is stored or embedded.
- Use **short topic queries**; pass the full question to the prompt, not the retriever.
- Keep `top_k_results` and `doc_content_chars_max` low to save tokens, or chunk the pages and use a vector store (section 5).
- Metadata keys are `title`, `summary`, `source`.